In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [2]:
CONTENT_DIR = "models/content/"
SENT_DIR    = "models/sentiment/"
DEVICE      = "cuda"

In [3]:
class UnifiedSemanticVector:
    def __init__(self, content_dir, sent_dir, device=DEVICE):
        self.device = device

        self.tokenizer = AutoTokenizer.from_pretrained(content_dir, use_fast=True)
        self.model_content = AutoModelForSequenceClassification.from_pretrained(content_dir).to(device).eval()
        self.model_sent    = AutoModelForSequenceClassification.from_pretrained(sent_dir).to(device).eval()

        self.labels = [
            "Алкоголь и табак",
            "Интернет магазины",
            "Компьютерные игры",
            "Криптомайнинг",
            "Порнография и секс",
            "Прокси и анонимайзеры",
            "Фильмы и видео онлайн",
            "Азартные игры",
            "Наркотики",
            "Суицид",
            "Депрессия",
            "Агрессия",
            "Разрешенный ресурс",
        ]

    @torch.inference_mode()
    def predict(self, text: str, max_length: int = 256, top_k: int = 5):
        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=max_length,
            padding=True,
            return_tensors="pt",
        ).to(self.device)

        pc = torch.softmax(self.model_content(**enc).logits[0], dim=-1)
        ps = torch.softmax(self.model_sent(**enc).logits[0], dim=-1)

        vector = torch.tensor([
            (pc[0] + pc[7]) * 0.8,        
            pc[1],                
            pc[2],                
            pc[3],                
            pc[4],                
            pc[5],                
            pc[8],                
            pc[9],                
            pc[10],               
            ps[1],                
            ps[2],                
            ps[3],                
            (pc[6] + ps[0]) * 0.5,        
        ], device=self.device)

        vector = vector / vector.sum()

        top_k = min(top_k, len(self.labels))
        values, indices = torch.topk(vector, k=top_k)

        top_categories = [
            {
                "label": self.labels[int(idx)],
                "score": float(val.item())
            }
            for val, idx in zip(values, indices)
        ]

        return {
            "vector": vector.detach().cpu().tolist(),
            "labels": self.labels,
            "top": top_categories,
        }

In [6]:
clf = UnifiedSemanticVector(CONTENT_DIR, SENT_DIR)

result = clf.predict("Слава Украине", top_k=5)

In [7]:
for i, item in enumerate(result["top"], 1):
    print(f"{i}. {item['label']} — {item['score']:.4f}")

1. Разрешенный ресурс — 0.9862
2. Суицид — 0.0036
3. Алкоголь и табак — 0.0019
4. Наркотики — 0.0016
5. Порнография и секс — 0.0014
